<a href="https://colab.research.google.com/github/lechefu/wealth-engine/blob/main/Copy_of_Final_Boss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import datetime
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
import logging
import math
import os
import concurrent.futures
from typing import List, Dict
from google.colab import data_table

warnings.simplefilter(action='ignore', category=FutureWarning)
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

# ====================== USER INPUTS ======================
# @title Apex Control Console
current_portfolio_value = 48822.34 # @param {type:"number"}
pivot_start_date = "2026-04-28" # @param {type:"date"}
pivot_start_value = 31000 # @param {type:"number"}
injection_amt = 10000 # @param {type:"number"}
strike_ticker = "poet" # @param {type:"string"}
catalyst_news = "Beat / Good Guidance" # @param ["Beat / Good Guidance", "Miss / Poor Guidance", "In-line / Mixed"] {type:"string"}
price_reaction = "Gapped Up" # @param ["Gapped Up", "Gapped Down", "Runaway Trend", "Flat or Dropped"] {type:"string"}
mass_scanner_on = True # @param {type:"boolean"}
mechanical_shield_base = 12 # @param {type:"number"}
use_sliding_shield = True # @param {type:"boolean"}
dynamic_universe_on = True # @param {type:"boolean"}
show_next_session_forecast = True # @param {type:"boolean"}

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def get_ipca_beta(ticker_metrics):
    char_vector = np.array([ticker_metrics['rsi'] / 100, ticker_metrics['vol_r'] / 10, ticker_metrics['vwap_g'] / 5])
    return float(np.dot(char_vector, [0.5, 0.3, 0.2]) * 2.0)

def get_ppi_metrics(ticker, benchmark="SPY"):
    try:
        ticker = ticker.strip().upper().replace(".", "")
        t = yf.Ticker(ticker)
        m = yf.Ticker(benchmark)
        h = t.history(period="60d")
        mh = m.history(period="60d")
        intra = t.history(period="1d", interval="5m")
        if h.empty or len(h) < 20: return None
        last = float(h['Close'].iloc[-1])
        prev = float(h['Close'].iloc[-2])
        growth_pct = ((last / prev) - 1) * 100
        delta = h['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rsi_val = float((100 - (100 / (1 + gain / loss))).iloc[-1])
        vwap_series = (intra['Close'] * intra['Volume']).cumsum() / intra['Volume'].cumsum() if not intra.empty else pd.Series([last])
        vwap_gap_val = ((last / float(vwap_series.iloc[-1])) - 1) * 100
        rets = h['Close'].pct_change().dropna()
        m_rets = mh['Close'].pct_change().dropna()
        beta = float(rets.cov(m_rets) / m_rets.var()) if m_rets.var() != 0 else 1.0
        bppi_val = growth_pct - (beta * ((float(mh['Close'].iloc[-1]) / float(mh['Close'].iloc[-2])) - 1) * 100)
        iwm = yf.Ticker("IWM").history(period="2d")
        ll = ((float(iwm['Close'].iloc[-1]) / float(iwm['Close'].iloc[-2])) - 1) * 100 if len(iwm) > 1 else 0.0
        tr = pd.concat([h['High'] - h['Low'], (h['High'] - h['Close'].shift(1)).abs(), (h['Low'] - h['Close'].shift(1)).abs()], axis=1).max(axis=1)
        vol_ratio = (float(tr.tail(14).mean()) / last) * 100
        consecutive_green = 0
        for ret in reversed(h['Close'].pct_change().tail(15).values):
            if ret > 0: consecutive_green += 1
            else: break
        risk_score = 0
        if bppi_val < 0: risk_score += 1
        if consecutive_green >= 6: risk_score += 3
        if rsi_val > 72: risk_score += 2
        risk_grade = "🛡️ LOW" if risk_score == 0 else ("⚠️ MED" if risk_score <= 2 else "🔴 HIGH")
        ppia_status = "🔴 ASYMMETRIC" if consecutive_green >= 6 else "🟢 HEALTHY"
        return {"last": last, "growth": growth_pct, "bppi": bppi_val, "vwap_g": vwap_gap_val, "ll": ll, "risk": risk_grade, "runup": consecutive_green, "vol_r": vol_ratio, "rsi": rsi_val, "ppia": ppia_status}
    except: return None

def calculate_sops_allocation(metrics, velocity_ref=0.01):
    meta_label = 1.0
    if metrics['vwap_g'] < 0: meta_label *= 0.7
    if metrics['ll'] < 0: meta_label *= 0.5
    regime_penalty = 1.5 if metrics['ll'] < 0 else 0.0
    ppia_penalty = 2.0 if metrics['runup'] >= 6 else 0.0
    conv_raw = (metrics['bppi'] * 0.45) + (velocity_ref * 12) - regime_penalty - ppia_penalty
    prob = sigmoid(conv_raw) * meta_label
    return prob, prob * 100

def get_performance_narrative(pm, prob, next_day_prob):
    if pm['bppi'] > 5 and pm['vwap_g'] > 1.5: return "Institutional Accumulation: Buying every dip."
    if prob >= 0.80: return f"🦁 APEX LION - Next-day conviction: {next_day_prob*100:.1f}%"
    return "🌱 Sideways/Neutral: No clear lead yet."

def get_earnings_directive(news, price, metrics=None):
    dynamic_shield = mechanical_shield_base
    if use_sliding_shield and metrics: dynamic_shield = max(5, mechanical_shield_base * (metrics['vol_r'] / 3.0))
    if news == "Beat / Good Guidance" and (price == "Gapped Up" or price == "Runaway Trend"): return ">>> 🦁 ALPHA HARVEST: Sell at open."
    elif news == "Miss / Poor Guidance" and price in ["Gapped Down", "Flat or Dropped"]: return f"!!! 🔴 TOTAL ABORT: Execute {dynamic_shield:.1f}% Shield."
    return "--- 🛡️ SHIELD LOCK: ATR Floor Active."

organic_growth_val = current_portfolio_value - injection_amt
_pivot = datetime.datetime.strptime(pivot_start_date, "%Y-%m-%d").date()
_today = datetime.datetime.now().date()
sessions_passed = max(1, int(np.busday_count(_pivot, _today)))
true_velocity = (organic_growth_val / pivot_start_value) ** (1.0 / sessions_passed) - 1.0

print(f"--- [STRIKE TELEMETRY: {strike_ticker.upper()}] ---")
m = get_ppi_metrics(strike_ticker)
if m:
    prob, alloc = calculate_sops_allocation(m, true_velocity)
    next_day_prob = min(0.98, prob * 0.93)
    shield_pct = max(5, mechanical_shield_base * (m['vol_r'] / 3.0)) if use_sliding_shield else mechanical_shield_base
    shield_floor_price = m['last'] * (1 - (shield_pct / 100))
    print(f"REGIME: {m['ppia']} | RUN-UP: {m['runup']} Days | PRICE: ${m['last']:.2f} | GROWTH: {m['growth']:+.2f}%)")
    print(f"BPPI: {m['bppi']:+.2f}% | IPCA BETA: {get_ipca_beta(m):.2f} | RSI: {m['rsi']:.1f} | VWAP GAP: {m['vwap_g']:+.2f}%)")
    print(f"RISK: {m['risk']} | SOPS ALLOC: {alloc:.1f}% | NEXT-DAY CONV: {next_day_prob*100:.1f}%)")
    print(f"SHIELD Floor: ${shield_floor_price:.2f} ({shield_pct:.1f}% below current)")
    print(f"DIRECTIVE: {get_earnings_directive(catalyst_news, price_reaction, m)}")
    print(f"NARRATIVE: {get_performance_narrative(m, prob, next_day_prob)}\n")

if mass_scanner_on:
    print("[SYSTEM] Rendering Interactive Scanner Feed...")
    master_list = ["NVDA","TSM","AVGO","ASML","AMD","MU","LRCX","KLAC","ADI","QCOM","TXN","MRVL","ARM","ENTG","TER","ONTO","ACLS","AEIS","FORM","AMBA","CRUS","SWKS","QRVO","LSCC","MCHP","MPWR","SMCI","ANET","PLTR","PATH","AI","UPST","SOFI","HOOD","COIN","MSTR","BTBT","MARA","IREN","CLSK","RIOT","HUT","BITF","LAZR","PLUG","FCEL","BE","BLDP","QS","ENVX","SLDP","MVIS","KTOS","AVAV","TXT","BA","GE","HON","ETN","EMR","ROK","AME","DOV","ITW","PH","IEX","WTS","FELE","GNRC","PWR","EME","FIX","APG","ACM","FLR","J","TTEK","STN","WLDN","NVEE","MYRG","ROAD","STRL","GVA","TPC","PRIM","DY","MDU","MTZ","POET","IPWR","PURE","AENT","SQNS","NVMI","KOPN","PXLW","FN","J","MX"]
    if dynamic_universe_on:
        extra = ["SOUN","BBAI","SERV","SYM","IONQ","QBTS","RGTI","QUBT","WRNT","LUNR","SMR","OKLO","NN","PSNL","BLI","DNA","TWST","CRSP","EDIT","NTLA","BEAM","VERV","FDMT","CGON","GLUE","KYMR","NRIX","STTK","ACET","VRDN","DAWN","MLYS","ABSI","CGEM","BCAB","TARS","OCUL","ANAB","KNSA","ITOS","PRME","CGTX","VKTX","ALT","MDGL","MNKD","GERN","SRRK","KROS","KURA","ZNTL","FULC","CRNX","INBX","ARVN","ACAD","AXSM","MCRB","NGNE","BIVI","LGVN","CLNN","LCTX","OCX","VRAX","BFRG","GCTK","NXL"]
        master_list = list(dict.fromkeys(master_list + extra))

    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=40) as executor:
        future_to_ticker = {executor.submit(get_ppi_metrics, t): t for t in master_list}
        for future in concurrent.futures.as_completed(future_to_ticker):
            t_sym = future_to_ticker[future]
            try:
                pm = future.result()
                if pm:
                    p_conv, _ = calculate_sops_allocation(pm, true_velocity)
                    nd_p = min(0.98, p_conv * 0.93)
                    status = "🦁📅 APEX" if nd_p >= 0.70 and p_conv >= 0.80 else ("🦁 LION" if p_conv >= 0.80 else "⏳ WAIT")
                    results.append({'TICKER': t_sym, 'STATUS': status, 'NEXT_DAY%': round(nd_p*100, 1), 'GROWTH%': round(pm['growth'], 2), 'BPPI%': round(pm['bppi'], 2), 'IPCA BETA': round(get_ipca_beta(pm), 2), 'RSI': round(pm['rsi'], 1), 'VWAP GAP%': round(pm['vwap_g'], 2), 'RISK': pm['risk']})
            except: continue

    if results:
        df_scan = pd.DataFrame(results)
        # Fix sorting: Define explicit category order so Apex/Lion always appear first
        df_scan['STATUS'] = pd.Categorical(df_scan['STATUS'], categories=["🦁📅 APEX", "🦁 LION", "⏳ WAIT"], ordered=True)
        df_scan = df_scan.sort_values(by=['STATUS', 'NEXT_DAY%'], ascending=[True, False])
        display(data_table.DataTable(df_scan, include_index=False, num_rows_per_page=15))
    else:
        print("[SYSTEM] No data found for the selected tickers. Please try running the scanner again.")

--- [STRIKE TELEMETRY: POET] ---
REGIME: 🟢 HEALTHY | RUN-UP: 2 Days | PRICE: $20.57 | GROWTH: +43.15%)
BPPI: +39.57% | IPCA BETA: 1.81 | RSI: 61.6 | VWAP GAP: +6.08%)
RISK: 🛡️ LOW | SOPS ALLOC: 100.0% | NEXT-DAY CONV: 93.0%)
SHIELD Floor: $10.83 (47.4% below current)
DIRECTIVE: >>> 🦁 ALPHA HARVEST: Sell at open.
NARRATIVE: Institutional Accumulation: Buying every dip.

[SYSTEM] Rendering Interactive Scanner Feed...


,TICKER,STATUS,NEXT_DAY%,GROWTH%,BPPI%,IPCA BETA,RSI,VWAP GAP%,RISK
44,BTBT,⏳ WAIT,72.7,4.93,2.37,1.33,67.1,2.48,🛡️ LOW
48,CLSK,⏳ WAIT,72.3,5.11,2.32,1.12,57.9,1.41,🛡️ LOW
150,MCRB,⏳ WAIT,72.3,3.48,2.32,1.08,49.0,1.84,🛡️ LOW
120,ACET,⏳ WAIT,72.2,4.21,2.31,1.22,58.7,3.32,🛡️ LOW
38,MARA,⏳ WAIT,71.2,4.24,2.16,1.13,62.6,0.91,🛡️ LOW
...,...,...,...,...,...,...,...,...,...
32,AI,🦁📅 APEX,75.6,4.39,2.80,0.91,56.2,0.62,🛡️ LOW
107,PSNL,🦁📅 APEX,75.5,5.01,2.78,1.14,57.6,0.72,🛡️ LOW
10,TSM,🦁📅 APEX,75.3,4.48,2.75,0.82,57.7,0.35,🛡️ LOW
29,HOOD,🦁📅 APEX,75.3,5.15,2.75,0.87,44.5,1.28,🛡️ LOW
